# 01. Baseline 데이터 점검 및 기준 모델

이 노트북은 해설서를 사용하지 않는 첫 번째 기준 실험입니다.
현재 단계에서는 사례집만 사용하며, DummyClassifier를 통해 최소 기준 성능을 측정합니다.

In [ ]:
!pip -q install pandas numpy scikit-learn openpyxl

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print("Loaded:", DATA_PATH)

In [ ]:
INPUT_COLUMNS = [
    "Product Name",
    "Main Ingredients (KR)",
    "Main Ingredients (EN)",
    "Product Form and Packaging",
    "Intended Use",
]
TARGET_COLUMN = "HS Code"

df = pd.read_excel(DATA_PATH, sheet_name="사례_전처리 완료")
required = INPUT_COLUMNS + [TARGET_COLUMN]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"필수 컬럼 누락: {missing_columns}")

for col in INPUT_COLUMNS:
    df[col] = df[col].fillna("").astype(str).str.strip()

df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne("")].reset_index(drop=True)

df["model_text"] = (
    "[PRODUCT] " + df["Product Name"] +
    " [INGREDIENTS_KR] " + df["Main Ingredients (KR)"] +
    " [INGREDIENTS_EN] " + df["Main Ingredients (EN)"] +
    " [FORM] " + df["Product Form and Packaging"] +
    " [USE] " + df["Intended Use"]
)

df["duplicate_group"] = pd.util.hash_pandas_object(
    df[INPUT_COLUMNS], index=False
).astype(str)

print("데이터 크기:", df.shape)
display(df[INPUT_COLUMNS + [TARGET_COLUMN]].head())

In [ ]:
missing_table = df[INPUT_COLUMNS].replace("", pd.NA).isna().sum().rename("missing_count").to_frame()
label_table = df[TARGET_COLUMN].value_counts().rename_axis("HS Code").reset_index(name="count")
group_table = df.groupby(TARGET_COLUMN)["duplicate_group"].nunique().rename("unique_input_groups").reset_index()
label_table = label_table.merge(group_table, on="HS Code", how="left")

print("결측치")
display(missing_table)
print("HS 코드별 사례 수")
display(label_table)
print("전체 HS 코드 수:", df[TARGET_COLUMN].nunique())
print("완전 중복 행 수:", df.duplicated(subset=INPUT_COLUMNS + [TARGET_COLUMN]).sum())
print("중복 입력 행 수:", df.duplicated(subset=INPUT_COLUMNS).sum())

## 평가 대상 클래스 선정

현재는 31개 클래스 중 사례 수가 5건 이상이고, 서로 다른 입력 그룹이 4개 이상인 클래스를 주 평가 대상으로 삼습니다. 나머지 희소 클래스는 삭제하지 않고 별도 분석 대상으로 보존합니다.

동일한 입력 정보가 여러 번 등장하면 `duplicate_group`으로 묶어 같은 Fold에 배치합니다.

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
eligible_labels = class_counts[class_counts >= 5].index
eval_df = df[df[TARGET_COLUMN].isin(eligible_labels)].copy()

group_counts = eval_df.groupby(TARGET_COLUMN)["duplicate_group"].nunique()
eligible_labels = group_counts[group_counts >= 4].index
eval_df = eval_df[eval_df[TARGET_COLUMN].isin(eligible_labels)].reset_index(drop=True)

print("전체 데이터:", len(df), "건 /", df[TARGET_COLUMN].nunique(), "개 코드")
print("주 평가 데이터:", len(eval_df), "건 /", eval_df[TARGET_COLUMN].nunique(), "개 코드")
print("별도 분석 데이터:", len(df) - len(eval_df), "건")
display(eval_df[TARGET_COLUMN].value_counts().rename("count").to_frame())

## Dummy baseline: 최빈 클래스 예측

DummyClassifier는 제품 정보를 학습하지 않고 학습 Fold에서 가장 많이 등장한 HS 코드만 예측합니다. 이후 실제 분류 모델이 넘어야 할 최소 기준점입니다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score

X = eval_df["model_text"]
y = eval_df[TARGET_COLUMN]
groups = eval_df["duplicate_group"]

cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    model = DummyClassifier(strategy="most_frequent")
    model.fit(X.iloc[train_idx].to_frame(), y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx].to_frame())

    fold_results.append({
        "fold": fold,
        "train_n": len(train_idx),
        "test_n": len(test_idx),
        "accuracy": accuracy_score(y.iloc[test_idx], pred),
        "macro_f1": f1_score(y.iloc[test_idx], pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y.iloc[test_idx], pred, average="weighted", zero_division=0),
    })

baseline_results = pd.DataFrame(fold_results)
display(baseline_results)
display(baseline_results[["accuracy", "macro_f1", "weighted_f1"]].agg(["mean", "std"]))

In [ ]:
baseline_results.to_csv("baseline_dummy_cv_results.csv", index=False, encoding="utf-8-sig")
label_table.to_csv("hs_code_data_audit.csv", index=False, encoding="utf-8-sig")
print("결과 CSV 저장 완료")